# Module 4: Multiagent Decision System

This module builds on top of the campus simulation developed in previous modules and introduces a **multiagent decision-making framework**.

Instead of a single rational agent, we now consider **two agents interacting in the same environment**, where outcomes depend on their **joint actions**.

### Objectives
- Understand how multiple agents affect decision-making
- Model both **competition and cooperation**
- Apply concepts from:
  - Multiagent environments
  - Non-cooperative game theory
  - Cooperative game theory
  - Collective decision-making

This represents a more realistic setting where agents must reason not only about the environment, but also about the behavior of others.

## Campus Environment (Reused from Module 3)

We reuse the same campus environment defined previously. This includes:

- Locations across campus
- Time slots throughout the day
- A graph representing movement connections between locations

This environment serves as the **shared world** in which both agents operate.

## Population Modeling and Behavior

We simulate a realistic campus population composed of:

- Athletes
- COPA students (arts)
- Regular students

Each student has:
- A type
- A program
- A set of friends

Their movement behavior depends on:
- Time-based preferences
- Social influence (friends)
- Program-specific tendencies

This creates a **dynamic and stochastic environment**, which is important for modeling uncertainty.

## Utility Model

Each agent evaluates actions using a **utility function** that represents cost.

The utility is based on:
- Movement cost
- Expected waiting time (based on crowd probability)
- Mismatch penalty (how suitable the location is)

Utility is defined as:
- Higher utility = better decision
- Utility = negative total cost

This ensures agents prefer:
- Short distances
- Low crowd levels
- Locations aligned with their profile

## Part 1: Properties of a Multiagent Environment

In this module, we move from a single-agent system to a **multiagent system**.

### Key Characteristics

- **Multiple agents**:
  - AthleteAgent
  - COPAAgent

- **Partial observability**:
  Agents rely on beliefs about crowd levels rather than exact information.

- **Stochastic environment**:
  Crowd behavior is uncertain and estimated using Monte Carlo simulation.

- **Mixed objectives**:
  Agents want good locations, but may compete for the same ones.

### Key Difference

Unlike single-agent systems, the outcome now depends on **joint actions**, meaning:
> One agent’s decision directly affects the other agent’s outcome.

## Part 2: Non-Cooperative Game Theory

In this section, each agent acts **independently**, trying to maximize its own utility.

We model the interaction as a **strategic game**:
- Each agent chooses an action (location)
- Payoffs depend on both agents’ actions

### Key Concepts

- **Best Response**:
  The best action given what the other agent does

- **Nash Equilibrium**:
  A stable outcome where no agent benefits from changing its decision alone

We construct a payoff matrix and analyze strategic behavior.

## Part 3: Cooperative Game Theory

Instead of competing, agents can **cooperate** to maximize total welfare.

We evaluate:
- Individual value (acting alone)
- Joint value (acting together)

### Fair Allocation

We use the **Shapley Value**, which:
- Distributes total value fairly
- Accounts for each agent’s contribution

This ensures both agents benefit proportionally from cooperation.

## Part 4: Collective Decision Making

Agents may also make decisions using **voting systems**.

We implement:
- **Plurality voting** (only top choice counts)
- **Borda count** (ranked scoring system)

We then evaluate outcomes using fairness metrics:
- Total welfare
- Minimum payoff
- Payoff inequality

This shows how different aggregation rules lead to different results.

## Demonstration

We now:
1. Estimate crowd probabilities using Monte Carlo simulation
2. Build the multiagent game
3. Compute:
   - Payoff matrix
   - Nash equilibria
   - Cooperative outcomes
   - Voting results

This ties together all components into a full decision system.

In [22]:
"""
MODULE 4: Multiagent Decision System
====================================
Built on top of Module 3 campus environment.

Goal:
- Two rational agents interact in the same campus world
- Outcomes depend on JOINT actions
- Includes:
  Part 1: properties of a multiagent environment
  Part 2: non-cooperative game theory
  Part 3: cooperative game theory
  Part 4: collective decision making
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import random
import math
from collections import Counter, defaultdict

# ============================================================
# Reused campus environment from Module 3
# ============================================================

LOCATIONS = [
    "LawrenceHall",
    "ThayerHall",
    "WestPenn",
    "StudentCenter",
    "Library",
    "VillagePark",
    "PlayHouse",
    "BoulevardStudios",
    "BoulevardAppartments",
    "OffCampus",
]

TIME_SLOTS = ["08:00", "10:00", "12:00", "14:00", "16:00", "18:00", "21:00"]
NUM_TIME_STEPS = len(TIME_SLOTS)

GRAPH: Dict[str, List[str]] = {
    "LawrenceHall": ["StudentCenter", "Library", "ThayerHall"],
    "ThayerHall": ["LawrenceHall", "WestPenn", "BoulevardStudios"],
    "WestPenn": ["ThayerHall", "VillagePark", "OffCampus"],
    "StudentCenter": ["LawrenceHall", "Library", "PlayHouse", "BoulevardAppartments"],
    "Library": ["LawrenceHall", "StudentCenter", "BoulevardStudios"],
    "VillagePark": ["WestPenn", "StudentCenter", "OffCampus"],
    "PlayHouse": ["StudentCenter", "BoulevardStudios"],
    "BoulevardStudios": ["Library", "PlayHouse", "ThayerHall", "BoulevardAppartments"],
    "BoulevardAppartments": ["StudentCenter", "BoulevardStudios", "OffCampus"],
    "OffCampus": ["WestPenn", "VillagePark", "BoulevardAppartments"],
}

def neighbors(loc: str) -> List[str]:
    return GRAPH.get(loc, [])

LOW, OK, HIGH = 0, 1, 2

def clamp_energy(e: int) -> int:
    return max(LOW, min(HIGH, e))

STUDENT_TYPES = ["athlete", "copa", "regular"]
ATHLETE_PROGRAMS = [
    "Baseball", "TrackAndField", "Wrestling", "Soccer", "ESports",
    "Softball", "Lacrosse", "Basketball", "Volleyball", "Golf"
]
COPA_PROGRAMS = [
    "Dance", "MusicalTheater", "Acting", "Film", "Animation", "ScreenWriting"
]

STUDENT_TYPE_PREFERENCES = {
    "athlete": [
        ["VillagePark", "StudentCenter"],
        ["StudentCenter", "Library"],
        ["StudentCenter", "BoulevardAppartments"],
        ["Library", "ThayerHall"],
        ["VillagePark", "WestPenn"],
        ["StudentCenter", "OffCampus"],
        LOCATIONS,
    ],
    "copa": [
        ["PlayHouse", "BoulevardStudios"],
        ["BoulevardStudios", "PlayHouse"],
        ["StudentCenter", "Library"],
        ["BoulevardStudios", "ThayerHall"],
        ["PlayHouse", "StudentCenter"],
        ["StudentCenter", "OffCampus"],
        LOCATIONS,
    ],
    "regular": [LOCATIONS] * NUM_TIME_STEPS,
}

PROGRAM_BOOST = {
    "TrackAndField": ["VillagePark"],
    "Soccer": ["VillagePark"],
    "Lacrosse": ["VillagePark"],
    "Baseball": ["VillagePark"],
    "Golf": ["VillagePark"],
    "Basketball": ["StudentCenter"],
    "Volleyball": ["StudentCenter"],
    "ESports": ["BoulevardAppartments", "StudentCenter"],
    "Dance": ["BoulevardStudios"],
    "MusicalTheater": ["PlayHouse"],
    "Acting": ["PlayHouse"],
    "Film": ["PlayHouse", "BoulevardStudios"],
    "Animation": ["BoulevardStudios"],
    "ScreenWriting": ["Library"],
}

@dataclass(frozen=True)
class Person:
    sid: str
    stype: str
    program: str
    friends: Tuple[str, ...]

def make_population(rng: random.Random) -> List[Person]:
    pop: List[Person] = []

    for i in range(6):
        prog = rng.choice(ATHLETE_PROGRAMS)
        pop.append(Person(sid=f"a{i}", stype="athlete", program=prog, friends=tuple()))

    for i in range(4):
        prog = rng.choice(COPA_PROGRAMS)
        pop.append(Person(sid=f"c{i}", stype="copa", program=prog, friends=tuple()))

    for i in range(10):
        pop.append(Person(sid=f"r{i}", stype="regular", program="None", friends=tuple()))

    ids_by_type: Dict[str, List[str]] = defaultdict(list)
    for p in pop:
        ids_by_type[p.stype].append(p.sid)

    pop2: List[Person] = []
    for p in pop:
        candidates = [x for x in ids_by_type[p.stype] if x != p.sid]
        k = rng.randint(0, min(2, len(candidates))) if candidates else 0
        friends = tuple(rng.sample(candidates, k)) if k else tuple()
        pop2.append(Person(p.sid, p.stype, p.program, friends))

    return pop2

def choose_location_for_person(
    person: Person,
    time_step: int,
    prev_positions: Dict[str, str],
    friend_boost: int,
    rng: random.Random,
) -> str:
    base = list(STUDENT_TYPE_PREFERENCES[person.stype][time_step])

    boosted: List[str] = []
    for fid in person.friends:
        if fid in prev_positions:
            boosted.append(prev_positions[fid])

    boosted += PROGRAM_BOOST.get(person.program, [])

    possible = base + boosted * friend_boost if boosted else base
    return rng.choice(possible)

def simulate_campus_day_positions(
    population: List[Person],
    friend_boost: int,
    rng: random.Random,
) -> List[Dict[str, str]]:
    trace: List[Dict[str, str]] = []
    prev: Dict[str, str] = {}

    for t in range(NUM_TIME_STEPS):
        step: Dict[str, str] = {}
        order = population[:]
        rng.shuffle(order)
        for p in order:
            step[p.sid] = choose_location_for_person(p, t, prev, friend_boost, rng)
        trace.append(step)
        prev = step

    return trace

def estimate_p_busy_table(
    num_traces: int = 2000,
    busy_threshold: float = 0.18,
    friend_boost: int = 2,
    seed: int = 7,
) -> List[Dict[str, float]]:
    rng = random.Random(seed)
    busy_counts: List[Counter] = [Counter() for _ in range(NUM_TIME_STEPS)]
    totals = [0 for _ in range(NUM_TIME_STEPS)]

    for _ in range(num_traces):
        pop = make_population(rng)
        tr = simulate_campus_day_positions(pop, friend_boost, rng)

        for t in range(NUM_TIME_STEPS):
            step = tr[t]
            n = max(1, len(step))
            counts = Counter(step.values())
            for L in LOCATIONS:
                frac = counts[L] / n
                if frac > busy_threshold:
                    busy_counts[t][L] += 1
            totals[t] += 1

    p_busy_table: List[Dict[str, float]] = []
    for t in range(NUM_TIME_STEPS):
        p_busy_table.append({L: busy_counts[t][L] / totals[t] for L in LOCATIONS})

    return p_busy_table

# ============================================================
# Shared utility model from Module 3
# ============================================================

MOVE_COST = 4.0
REST_COST = 0.0

def expected_wait(p_busy: float) -> float:
    wait_busy, wait_not = 12.0, 3.0
    return p_busy * wait_busy + (1 - p_busy) * wait_not

MISMATCH_PENALTY: Dict[Tuple[str, str], float] = {
    ("athlete", "VillagePark"): 0.0,
    ("athlete", "StudentCenter"): 2.0,
    ("athlete", "PlayHouse"): 4.0,
    ("athlete", "BoulevardStudios"): 4.0,
    ("athlete", "Library"): 2.0,
    ("athlete", "OffCampus"): 3.0,

    ("copa", "PlayHouse"): 0.0,
    ("copa", "BoulevardStudios"): 0.0,
    ("copa", "StudentCenter"): 1.5,
    ("copa", "Library"): 1.5,
    ("copa", "VillagePark"): 4.0,
    ("copa", "OffCampus"): 2.5,
}

def mismatch(student_type: str, loc: str) -> float:
    return MISMATCH_PENALTY.get((student_type, loc), 1.0)

# ============================================================
# Module 4: Multiagent system
# ============================================================

"""
PART 1 (18.1): Properties of a multiagent environment

Environment properties:
- At least two agents:
    Agent A = athlete
    Agent B = copa
- Partial observability:
    Agents know time and a belief about crowd, but not the true crowd state with certainty
- Stochastic:
    Crowd beliefs come from Monte Carlo simulation
- Mixed motives:
    Both agents prefer low-cost, low-crowd destinations,
    but they may compete for the same attractive location
- Differs from single-agent:
    Utility now depends on JOINT actions, not only one agent's action
"""

@dataclass
class AgentProfile:
    name: str
    agent_type: str

@dataclass
class MultiAgentState:
    time_slot: str
    belief_busy: Dict[str, float]
    athlete_energy: int = OK
    copa_energy: int = OK

# Small strategic action set for readable game matrix
GAME_ACTIONS = ["StudentCenter", "Library", "VillagePark"]

CONGESTION_PENALTY = 4.0  # if both go to same place
COORDINATION_BONUS = 1.0  # optional small bonus if they coordinate efficiently apart

def base_payoff(
    agent_type: str,
    action_location: str,
    belief_busy: Dict[str, float]
) -> float:
    """
    Base payoff before strategic interaction:
    negative total time/cost
    """
    p_busy = belief_busy[action_location]
    total_cost = MOVE_COST + expected_wait(p_busy) + mismatch(agent_type, action_location)
    return -total_cost

def joint_payoff(
    agent_i_type: str,
    action_i: str,
    action_j: str,
    belief_busy: Dict[str, float]
) -> float:
    """
    Multiagent payoff:
    - base utility from own choice
    - congestion penalty if both choose same location
    - optional small coordination bonus if they avoid congestion
    """
    u = base_payoff(agent_i_type, action_i, belief_busy)

    if action_i == action_j:
        u -= CONGESTION_PENALTY
    else:
        u += COORDINATION_BONUS

    return u

# ============================================================
# Part 2 (18.2): Non-Cooperative Game Theory
# ============================================================

def build_payoff_matrix(
    state: MultiAgentState,
    agentA: AgentProfile,
    agentB: AgentProfile
) -> Dict[Tuple[str, str], Tuple[float, float]]:
    """
    Returns payoff matrix:
      matrix[(aA, aB)] = (uA, uB)
    """
    matrix: Dict[Tuple[str, str], Tuple[float, float]] = {}

    for aA in GAME_ACTIONS:
        for aB in GAME_ACTIONS:
            uA = joint_payoff(agentA.agent_type, aA, aB, state.belief_busy)
            uB = joint_payoff(agentB.agent_type, aB, aA, state.belief_busy)
            matrix[(aA, aB)] = (uA, uB)

    return matrix

def best_responses_A(matrix: Dict[Tuple[str, str], Tuple[float, float]]) -> Dict[str, List[str]]:
    """
    For each action of B, return A's best responses.
    """
    br: Dict[str, List[str]] = {}
    for aB in GAME_ACTIONS:
        vals = {aA: matrix[(aA, aB)][0] for aA in GAME_ACTIONS}
        best_val = max(vals.values())
        br[aB] = [aA for aA, v in vals.items() if abs(v - best_val) < 1e-12]
    return br

def best_responses_B(matrix: Dict[Tuple[str, str], Tuple[float, float]]) -> Dict[str, List[str]]:
    """
    For each action of A, return B's best responses.
    """
    br: Dict[str, List[str]] = {}
    for aA in GAME_ACTIONS:
        vals = {aB: matrix[(aA, aB)][1] for aB in GAME_ACTIONS}
        best_val = max(vals.values())
        br[aA] = [aB for aB, v in vals.items() if abs(v - best_val) < 1e-12]
    return br

def find_nash_equilibria(matrix: Dict[Tuple[str, str], Tuple[float, float]]) -> List[Tuple[str, str]]:
    brA = best_responses_A(matrix)
    brB = best_responses_B(matrix)

    ne: List[Tuple[str, str]] = []
    for aA in GAME_ACTIONS:
        for aB in GAME_ACTIONS:
            if aA in brA[aB] and aB in brB[aA]:
                ne.append((aA, aB))
    return ne

# ============================================================
# Part 3 (18.3): Cooperative Game Theory
# ============================================================

def coalition_value_single(
    agent: AgentProfile,
    belief_busy: Dict[str, float]
) -> float:
    """
    Value of singleton coalition: best payoff agent can get alone.
    No congestion penalty because no other agent exists.
    """
    vals = [base_payoff(agent.agent_type, a, belief_busy) for a in GAME_ACTIONS]
    return max(vals)

def coalition_value_pair(
    agentA: AgentProfile,
    agentB: AgentProfile,
    belief_busy: Dict[str, float]
) -> Tuple[float, Tuple[str, str]]:
    """
    Value of coalition {A,B}: maximum joint welfare if they coordinate.
    """
    best_val = -float("inf")
    best_joint = (GAME_ACTIONS[0], GAME_ACTIONS[0])

    for aA in GAME_ACTIONS:
        for aB in GAME_ACTIONS:
            uA = joint_payoff(agentA.agent_type, aA, aB, belief_busy)
            uB = joint_payoff(agentB.agent_type, aB, aA, belief_busy)
            total = uA + uB
            if total > best_val:
                best_val = total
                best_joint = (aA, aB)

    return best_val, best_joint

def shapley_two_players(
    agentA: AgentProfile,
    agentB: AgentProfile,
    belief_busy: Dict[str, float]
) -> Dict[str, float]:
    """
    Shapley value for 2-player coalition game:
      φA = 1/2 [ v({A}) + (v({A,B}) - v({B})) ]
      φB = 1/2 [ v({B}) + (v({A,B}) - v({A})) ]
    """
    vA = coalition_value_single(agentA, belief_busy)
    vB = coalition_value_single(agentB, belief_busy)
    vAB, _ = coalition_value_pair(agentA, agentB, belief_busy)

    phiA = 0.5 * (vA + (vAB - vB))
    phiB = 0.5 * (vB + (vAB - vA))

    return {
        "v({A})": vA,
        "v({B})": vB,
        "v({A,B})": vAB,
        agentA.name: phiA,
        agentB.name: phiB,
    }

# ============================================================
# Part 4 (18.4): Collective Decision Making
# ============================================================

def rank_actions_for_agent(agent: AgentProfile, belief_busy: Dict[str, float]) -> List[str]:
    """
    Rank strategic actions from best to worst for one agent.
    Uses base payoff only (collective preference over destinations).
    """
    scored = [(base_payoff(agent.agent_type, a, belief_busy), a) for a in GAME_ACTIONS]
    scored.sort(reverse=True)
    return [a for _, a in scored]

def plurality_vote(rankings: Dict[str, List[str]]) -> Tuple[str, Dict[str, int]]:
    """
    Each agent votes for top choice only.
    """
    votes = {a: 0 for a in GAME_ACTIONS}
    for _, ranking in rankings.items():
        votes[ranking[0]] += 1
    winner = max(votes, key=votes.get)
    return winner, votes

def borda_vote(rankings: Dict[str, List[str]]) -> Tuple[str, Dict[str, int]]:
    """
    3 actions:
      top -> 2 points
      second -> 1 point
      third -> 0 points
    """
    scores = {a: 0 for a in GAME_ACTIONS}
    for _, ranking in rankings.items():
        for points, action in zip([2, 1, 0], ranking):
            scores[action] += points
    winner = max(scores, key=scores.get)
    return winner, scores

def fairness_of_collective_choice(
    winner: str,
    agents: List[AgentProfile],
    belief_busy: Dict[str, float]
) -> Dict[str, float]:
    """
    Evaluate a collective decision by:
    - total welfare
    - minimum individual payoff
    - gap between best and worst
    """
    payoffs = [base_payoff(agent.agent_type, winner, belief_busy) for agent in agents]
    total = sum(payoffs)
    min_pay = min(payoffs)
    gap = max(payoffs) - min(payoffs)

    return {
        "total_welfare": total,
        "min_individual_payoff": min_pay,
        "payoff_gap": gap,
    }

# ============================================================
# Demo / report output
# ============================================================

def print_payoff_matrix(matrix: Dict[Tuple[str, str], Tuple[float, float]]) -> None:
    print("Payoff matrix: rows = Athlete actions, cols = COPA actions")
    print("Entry format: (uAthlete, uCOPA)\n")

    header = " " * 18 + "".join(f"{aB:>28s}" for aB in GAME_ACTIONS)
    print(header)

    for aA in GAME_ACTIONS:
        row = f"{aA:>18s}"
        for aB in GAME_ACTIONS:
            uA, uB = matrix[(aA, aB)]
            row += f"{f'({uA:.2f},{uB:.2f})':>28s}"
        print(row)

if __name__ == "__main__":
    print("=== Building belief table p_busy[L,t] from Module 3 world ===")
    p_busy_table = estimate_p_busy_table(
        num_traces=2000,
        busy_threshold=0.18,
        friend_boost=2,
        seed=7
    )

    t_demo = TIME_SLOTS.index("16:00")
    belief_busy = p_busy_table[t_demo]

    print(f"\nBelief snapshot at {TIME_SLOTS[t_demo]}:")
    for L in GAME_ACTIONS:
        print(f"  P(Busy({L}) at {TIME_SLOTS[t_demo]}) ≈ {belief_busy[L]:.3f}")

    # ---------------- Part 1 ----------------
    print("\n\n==============================")
    print("PART 1: Properties of Multiagent Environment")
    print("==============================")
    print("Agents:")
    print("  Agent A = athlete")
    print("  Agent B = copa")
    print("\nObservability:")
    print("  Partially observable: agents know beliefs about crowd, not the exact true crowd state.")
    print("\nDeterminism:")
    print("  Stochastic: crowd beliefs come from Monte Carlo estimates.")
    print("\nGoals:")
    print("  Mixed: both want good destinations, but congestion makes them compete.")
    print("\nDifference from single-agent:")
    print("  Each agent's payoff depends on the OTHER agent's action, not just its own action.")

    # Profiles
    athlete = AgentProfile(name="AthleteAgent", agent_type="athlete")
    copa = AgentProfile(name="COPAAgent", agent_type="copa")
    state = MultiAgentState(time_slot=TIME_SLOTS[t_demo], belief_busy=belief_busy)

    # ---------------- Part 2 ----------------
    print("\n\n==============================")
    print("PART 2: Non-Cooperative Game Theory")
    print("==============================")

    matrix = build_payoff_matrix(state, athlete, copa)
    print_payoff_matrix(matrix)

    brA = best_responses_A(matrix)
    brB = best_responses_B(matrix)
    ne = find_nash_equilibria(matrix)

    print("\nBest responses of AthleteAgent:")
    for aB in GAME_ACTIONS:
        print(f"  If COPA chooses {aB}, Athlete best responds with {brA[aB]}")

    print("\nBest responses of COPAAgent:")
    for aA in GAME_ACTIONS:
        print(f"  If Athlete chooses {aA}, COPA best responds with {brB[aA]}")

    print("\nNash equilibria:")
    if ne:
        for eq in ne:
            print(" ", eq)
    else:
        print("  No pure Nash equilibrium found.")

    # ---------------- Part 3 ----------------
    print("\n\n==============================")
    print("PART 3: Cooperative Game Theory")
    print("==============================")

    vAB, best_joint = coalition_value_pair(athlete, copa, belief_busy)
    shapley = shapley_two_players(athlete, copa, belief_busy)

    print(f"Best joint action under cooperation: {best_joint}")
    print(f"Coalition value v({{A,B}}) = {vAB:.3f}")

    print("\nShapley allocation:")
    for k, v in shapley.items():
        print(f"  {k}: {v:.3f}")

    print("\nWhy the allocation is reasonable:")
    print("  Each agent receives value based on its standalone ability and its marginal contribution")
    print("  to the coalition. This is a standard fair allocation for 2-player cooperative games.")

    # ---------------- Part 4 ----------------
    print("\n\n==============================")
    print("PART 4: Collective Decision Making")
    print("==============================")

    rankings = {
        athlete.name: rank_actions_for_agent(athlete, belief_busy),
        copa.name: rank_actions_for_agent(copa, belief_busy),
    }

    print("Agent rankings:")
    for name, ranking in rankings.items():
        print(f"  {name}: {ranking}")

    plurality_winner, plurality_votes = plurality_vote(rankings)
    borda_winner, borda_scores = borda_vote(rankings)

    print("\nPlurality result:")
    print("  Votes:", plurality_votes)
    print("  Winner:", plurality_winner)

    print("\nBorda result:")
    print("  Scores:", borda_scores)
    print("  Winner:", borda_winner)

    fairness_plurality = fairness_of_collective_choice(plurality_winner, [athlete, copa], belief_busy)
    fairness_borda = fairness_of_collective_choice(borda_winner, [athlete, copa], belief_busy)

    print("\nFairness comparison:")
    print(f"  Plurality -> {fairness_plurality}")
    print(f"  Borda     -> {fairness_borda}")

    print("\n--- Markdown-ready explanation (paste into notebook) ---\n")
    print(f"""
### Part 1 (Properties of a Multiagent Environment)
- The environment contains **two agents**:
  - AthleteAgent
  - COPAAgent
- Each agent has its own actions and utilities.
- **Observability**: partial, because agents reason using belief about crowd rather than exact hidden state.
- **Determinism**: stochastic, because beliefs come from uncertain crowd estimates.
- **Goals**: mixed. Agents both want good locations, but congestion creates conflict.
- This differs from a single-agent problem because each agent's payoff depends on the **joint action** of both agents.

### Part 2 (Non-Cooperative Game Theory)
- Players: AthleteAgent, COPAAgent
- Actions: {GAME_ACTIONS}
- Payoffs: based on expected cost (walking + waiting + mismatch) plus congestion penalty if both choose the same location.
- Best responses were computed for each player.
- Nash equilibrium was identified by checking mutual best responses.

### Part 3 (Cooperative Game Theory)
- Coalition values:
  - v(A): best payoff Athlete can guarantee alone
  - v(B): best payoff COPA can guarantee alone
  - v(A,B): best total welfare if they coordinate
- Fair allocation used: **Shapley value**
- This is reasonable because it rewards both standalone ability and marginal contribution.

### Part 4 (Collective Decisions)
- The agents rank candidate destinations.
- Two aggregation rules were implemented:
  - **Plurality**
  - **Borda Count**
- Outcomes are compared using fairness criteria:
  - total welfare
  - minimum individual payoff
  - payoff gap
""".strip())

=== Building belief table p_busy[L,t] from Module 3 world ===

Belief snapshot at 16:00:
  P(Busy(StudentCenter) at 16:00) ≈ 0.279
  P(Busy(Library) at 16:00) ≈ 0.115
  P(Busy(VillagePark) at 16:00) ≈ 0.553


PART 1: Properties of Multiagent Environment
Agents:
  Agent A = athlete
  Agent B = copa

Observability:
  Partially observable: agents know beliefs about crowd, not the exact true crowd state.

Determinism:
  Stochastic: crowd beliefs come from Monte Carlo estimates.

Goals:
  Mixed: both want good destinations, but congestion makes them compete.

Difference from single-agent:
  Each agent's payoff depends on the OTHER agent's action, not just its own action.


PART 2: Non-Cooperative Game Theory
Payoff matrix: rows = Athlete actions, cols = COPA actions
Entry format: (uAthlete, uCOPA)

                                 StudentCenter                     Library                 VillagePark
     StudentCenter             (-15.51,-15.01)              (-10.51,-8.54)             (-10.